# Analysis of the Metroborgo Project Questionnaire

## Analysis objective

This notebook analyses the results of the questionnaire about the Metroborgo project and perceptions of Montalto delle Marche.

The objective is to describe separately:

* residents and regular visitors to the area;
* non-resident or occasional visitors.

The analysis considers respondents' profiles, the characteristics of their visit experience, perceptions of the area, the effects attributed to the project, and selected economic aspects.

## Structure of the analysis

The procedure includes:

1. loading and checking the dataset;
2. separating residents from visitors;
3. removing irrelevant columns and questionnaires without useful responses;
4. checking partially completed questionnaires;
5. descriptive analysis of the responses;
6. calculating frequencies, percentages, means, and medians;
7. grouping statements into thematic areas.

Percentages are calculated using the number of valid responses available for each question. Missing responses are not replaced with estimated values.

## Notebook organisation

The notebook is divided into two main sections:

* visitor analysis;
* analysis of residents and regular visitors.

Final interpretations and charts are developed separately in the final presentation.


## Importing libraries and loading the data

This section imports the Pandas library, which is used to read, organise, and analyse the data.  
The CSV file containing the questionnaire responses is then loaded.


In [ ]:
# Import the Pandas library
import pandas as pd

# Load the CSV file
df = pd.read_csv("results-survey-07.csv")

# Display the first five rows of the dataset
df.head()

## Initial dataset check

I verify that the file has been loaded correctly and check the total number of questionnaires.


In [ ]:
# Check the total number of responses in the dataset
numero_risposte = df.shape[0]

print("Numero totale di risposte:", numero_risposte)

# Display the first five rows to verify that the data loaded correctly
df.head()

## Sample division

The questionnaire divides respondents according to the following question:

**“Do you live in Montalto delle Marche or visit it regularly?”**

The responses are used to create two groups:

- **Residents or regular visitors**: “Yes” responses
- **Non-residents or occasional visitors**: “No” responses

The division retains all responses associated with each participant, since each row in the dataset represents one complete questionnaire.


In [ ]:
# Name of the column used to divide the sample
colonna_residenza = (
    "Vivi a Montalto delle Marche o la frequenti abitualmente?"
)

# Check the responses in the column
df[colonna_residenza].value_counts(dropna=False)

## Removing missing responses and creating the groups

The two questionnaires with no answer to the grouping question are excluded from the comparative analysis.

The valid sample is then divided into two DataFrames:

* residents or regular visitors;
* non-residents or occasional visitors.


In [ ]:
# Name of the column used to divide the sample
colonna_residenza = (
    "Vivi a Montalto delle Marche o la frequenti abitualmente?"
)

# Create a copy of the dataset, removing only rows
# with no answer to the grouping question
df_valido = df.dropna(subset=[colonna_residenza]).copy()

# Create the DataFrame for residents or regular visitors
df_residenti = df_valido[
    df_valido[colonna_residenza] == "Sì"
].copy()

# Create the DataFrame for non-residents or occasional visitors
df_turisti = df_valido[
    df_valido[colonna_residenza] == "No"
].copy()

# Reset the indices of the two DataFrames
df_residenti.reset_index(drop=True, inplace=True)
df_turisti.reset_index(drop=True, inplace=True)

# Check the number of responses in the two groups
print("Risposte valide totali:", len(df_valido))
print("Residents or regular visitors:", len(df_residenti))
print("Non-residents or occasional visitors:", len(df_turisti))

## Initial check and cleaning of the tourist DataFrame

Before analysing the responses, the structure of the tourist DataFrame is checked.

In particular:

1. all columns are displayed;
2. the number of responses in each column is counted;
3. completely empty columns are identified and removed;
4. rows without useful responses are identified.

The original `df_turisti` DataFrame is retained, while the cleaning operations are applied to a copy.


In [ ]:
# Print all headers in the tourist DataFrame
for numero, colonna in enumerate(df_turisti.columns, start=1):
    print(f"{numero}. {colonna}")

print("\nTotal number of columns:", len(df_turisti.columns))

### Number of responses by column

For each question, the number of non-missing values is counted. This check makes it possible to distinguish used columns from completely empty ones.


In [ ]:
# Count non-missing values in each column
conteggio_risposte_turisti = df_turisti.notna().sum()

# Convert the result into a more readable table
riepilogo_colonne_turisti = (
    conteggio_risposte_turisti
    .reset_index()
)

riepilogo_colonne_turisti.columns = [
    "Column",
    "Number of responses"
]

# Add the number of missing values
riepilogo_colonne_turisti["Missing values"] = (
    len(df_turisti)
    - riepilogo_colonne_turisti["Number of responses"]
)

riepilogo_colonne_turisti

### Removing columns not relevant to the tourist group

The tourist DataFrame contains some questions intended exclusively for residents. Since these columns contain no responses in the group being analysed, they are removed.

No rows are removed at this stage, because some questionnaires may have been only partially completed.


In [ ]:
# Identify completely empty columns in the tourist group
colonne_vuote_turisti = df_turisti.columns[
    df_turisti.isna().all()
].tolist()

# Create a copy, removing only completely empty columns
df_turisti_pulito = df_turisti.drop(
    columns=colonne_vuote_turisti
).copy()

print("Original columns:", df_turisti.shape[1])
print("Completely empty columns removed:", len(colonne_vuote_turisti))
print("Remaining columns:", df_turisti_pulito.shape[1])

### Checking the remaining columns

After removing the completely empty columns, the 51 remaining columns are examined by displaying their full names and the number of responses they contain.

This check makes it possible to distinguish technical information, questions addressed to tourists, sociodemographic data, and any anomalous columns.


In [ ]:
# Display the full text contained in the table columns
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

# Create a summary of the remaining columns
riepilogo_turisti_pulito = pd.DataFrame({
    "Column": df_turisti_pulito.columns,
    "Number of responses": df_turisti_pulito.notna().sum().values,
    "Missing values": df_turisti_pulito.isna().sum().values
})

riepilogo_turisti_pulito

### Removing irrelevant and technical columns

The following are removed from the tourist DataFrame:

* the five columns concerning changes in property value, which are intended for residents;
* technical variables generated by the data-collection system;
* the question used to divide the sample, since all participants in this DataFrame already belong to the non-resident group.

The response ID is retained to allow checks on individual questionnaires.


In [ ]:
# Technical columns not required for the analysis
colonne_tecniche_turisti = [
    "Data invio",
    "Ultima pagina",
    "Lingua iniziale",
    "Seme",
    "Data di inizio",
    "Data dell'ultima azione",
    colonna_residenza
]

# Identify columns concerning changes in property value,
# which belong to the resident questionnaire
colonne_patrimonio = [
    colonna
    for colonna in df_turisti_pulito.columns
    if colonna.startswith(
        "A seguito degli investimenti realizzati con il progetto Metroborgo, "
        "ritieni che il valore del tuo patrimonio"
    )
]

# Combine all columns to be removed
colonne_da_eliminare = colonne_tecniche_turisti + colonne_patrimonio

# Keep only columns that are actually present
colonne_da_eliminare = [
    colonna
    for colonna in colonne_da_eliminare
    if colonna in df_turisti_pulito.columns
]

# Remove the columns from the cleaned copy
df_turisti_pulito = df_turisti_pulito.drop(
    columns=colonne_da_eliminare
).copy()

print("Columns removed:", len(colonne_da_eliminare))
print("Dimensioni del DataFrame pulito:", df_turisti_pulito.shape)

### Checking questionnaire completion levels

For each questionnaire, the number of answers actually provided is calculated.
The response ID is excluded from the count because it is not a questionnaire item.

This check helps identify complete, partial, or almost empty questionnaires before deciding whether any rows should be removed.


In [ ]:
# Select only columns containing questionnaire items
colonne_analisi_turisti = [
    colonna
    for colonna in df_turisti_pulito.columns
    if colonna != "ID risposta"
]

# Calculate the number of completed responses for each questionnaire
completamento_turisti = pd.DataFrame({
    "ID risposta": df_turisti_pulito["ID risposta"],
    "Completed responses": df_turisti_pulito[
        colonne_analisi_turisti
    ].notna().sum(axis=1)
})

# Also calculate the number of missing responses
completamento_turisti["Missing responses"] = (
    len(colonne_analisi_turisti)
    - completamento_turisti["Completed responses"]
)

# Sort from the least complete to the most complete questionnaires
completamento_turisti = completamento_turisti.sort_values(
    by="Completed responses"
)

completamento_turisti

### Removing questionnaires without responses

Eleven questionnaires were identified as containing no answers to the questions useful for the tourist analysis.

These questionnaires are excluded, while partially completed questionnaires are retained for subsequent assessment.


In [ ]:
# Identify rows with no answers to the analytical questions
indici_questionari_vuoti = completamento_turisti[
    completamento_turisti["Completed responses"] == 0
].index

print("Completely empty questionnaires:",
      len(indici_questionari_vuoti))

# Remove only questionnaires without responses
df_turisti_pulito = (
    df_turisti_pulito
    .drop(index=indici_questionari_vuoti)
    .reset_index(drop=True)
)

print("Updated dimensions:",
      df_turisti_pulito.shape)

### Checking partially completed questionnaires

After removing completely empty questionnaires, submissions containing only five responses are examined.

This check makes it possible to identify which sections were completed and to decide whether the questionnaires should be retained or excluded from the analysis.


In [ ]:
# Identify questionnaires containing only five completed responses
id_questionari_parziali = [120, 199]

# Select the two questionnaires
questionari_parziali = df_turisti_pulito[
    df_turisti_pulito["ID risposta"].isin(id_questionari_parziali)
]

# For each questionnaire, display only completed columns
for _, riga in questionari_parziali.iterrows():
    print("\n" + "=" * 70)
    print("Response ID:", riga["ID risposta"])
    print("=" * 70)

    risposte_presenti = riga.dropna()

    for colonna, risposta in risposte_presenti.items():
        if colonna != "ID risposta":
            print(f"\n{colonna}")
            print("Response:", risposta)

### Checking open-ended responses

Text responses are examined before the analysis to identify test questionnaires, invalid submissions, or notes requiring the exclusion of a response.

Each comment is displayed together with the questionnaire ID.


In [ ]:
# Columns containing open-ended responses
colonne_commenti = [
    "Cosa ti ha colpito di più di questa esperienza e perché?",
    "Pensieri liberi. C’è qualcos’altro che vorresti aggiungere?"
]

# Select questionnaires containing at least one comment
commenti_turisti = df_turisti_pulito.loc[
    df_turisti_pulito[colonne_commenti].notna().any(axis=1),
    ["ID risposta"] + colonne_commenti
]

# Print comments without truncation
for _, riga in commenti_turisti.iterrows():
    print("\n" + "=" * 70)
    print("Response ID:", riga["ID risposta"])

    for colonna in colonne_commenti:
        if pd.notna(riga[colonna]):
            print(f"\n{colonna}")
            print(riga[colonna])

### Excluding a test questionnaire

Response ID 128 explicitly contains the note “do not consider” and appears to be a test submission.

The questionnaire is therefore excluded from the tourist DataFrame and from all subsequent analyses.


In [ ]:
# ID of the test questionnaire
id_da_escludere = [128]

# Exclude the questionnaire from the general tourist DataFrame
df_turisti_pulito = df_turisti_pulito[
    ~df_turisti_pulito["ID risposta"].isin(id_da_escludere)
].copy()

df_turisti_pulito.reset_index(drop=True, inplace=True)

print("Remaining tourist questionnaires:", len(df_turisti_pulito))

In [ ]:
# Incomplete questionnaires to exclude
id_questionari_incompleti = [120, 199]

# Remove the questionnaires from the tourist DataFrame
df_turisti_pulito = df_turisti_pulito[
    ~df_turisti_pulito["ID risposta"].isin(
        id_questionari_incompleti
    )
].copy()

# Reset the index
df_turisti_pulito.reset_index(drop=True, inplace=True)

# Check the final result
print("Questionari turistici validi:",
      len(df_turisti_pulito))

print("DataFrame dimensions:",
      df_turisti_pulito.shape)

## Sociodemographic profile of visitors

This section describes the composition of the visitor sample using four variables:

* gender;
* age group;
* educational qualification;
* occupation.

For each variable, the number of respondents and the percentage of valid responses are calculated.


In [ ]:
# Sociodemographic variables to analyse
variabili_profilo = [
    "Genere",
    "Età:",
    "Titolo di studio",
    "Professione"
]

# Function for creating a table with counts and percentages
def tabella_percentuali(dataframe, colonna):
    tabella = (
        dataframe[colonna]
        .dropna()
        .value_counts()
        .rename_axis(colonna)
        .reset_index(name="Number of responses")
    )

    tabella["Percentuale"] = (
        tabella["Number of responses"]
        / tabella["Number of responses"].sum()
        * 100
    ).round(1)

    return tabella

# Create and display a table for each variable
for variabile in variabili_profilo:
    print("\n" + "=" * 70)
    print(variabile.upper())
    print("=" * 70)

    display(
        tabella_percentuali(
            df_turisti_pulito,
            variabile
        )
    )

## Visit characteristics

This section analyses the main characteristics of visits to Montalto delle Marche.

The analysis considers:

* whether it is the respondent's first visit;
* the duration or type of stay;
* the main reason for the visit;
* who accompanied the respondent;
* participation in events.

For each question, the number of responses and the corresponding percentage are shown, calculated using the total number of valid responses.


In [ ]:
# Variables concerning visit characteristics
variabili_visita = [
    "È la tua prima visita a Montalto delle Marche?",
    "Che tipo di visita hai o stai realizzando?",
    "Qual'è il motivo principale della tua visita?",
    "Con chi hai visitato Montalto delle Marche?",
    "Hai partecipato ad eventi?"
]

# Create and display a table for each variable
for variabile in variabili_visita:
    print("\n" + "=" * 70)
    print(variabile.upper())
    print("=" * 70)

    display(
        tabella_percentuali(
            df_turisti_pulito,
            variabile
        )
    )

## Evaluation of the tourist experience

This section analyses statements concerning the perceived quality and effects of the visit experience.

Responses are given on a scale from 1 to 7. For each statement, the following are calculated:

* the number of valid responses;
* the mean score;
* the median;
* the percentage of positive ratings, corresponding to scores from 5 to 7.

Higher mean values indicate a more positive perception of the experience.


In [ ]:
# Identify columns containing ratings on a 1-to-7 scale
colonne_valutazioni = [
    colonna
    for colonna in df_turisti_pulito.columns
    if colonna.startswith(
        "Quanto sei d’accordo con le seguenti affermazioni?"
    )
]

# Create a summary table
riepilogo_valutazioni = []

for colonna in colonne_valutazioni:
    risposte = pd.to_numeric(
        df_turisti_pulito[colonna],
        errors="coerce"
    ).dropna()

    # Extract only the statement text enclosed in square brackets
    affermazione = colonna.split("[", 1)[-1].rstrip("]")

    riepilogo_valutazioni.append({
        "Statement": affermazione,
        "Valid responses": len(risposte),
        "Mean": risposte.mean(),
        "Median": risposte.median(),
        "Positive ratings (%)": (
            risposte.ge(5).mean() * 100
        )
    })

riepilogo_valutazioni = pd.DataFrame(
    riepilogo_valutazioni
)

# Round the results
riepilogo_valutazioni["Mean"] = (
    riepilogo_valutazioni["Mean"].round(2)
)

riepilogo_valutazioni["Median"] = (
    riepilogo_valutazioni["Median"].round(1)
)

riepilogo_valutazioni["Positive ratings (%)"] = (
    riepilogo_valutazioni[
        "Positive ratings (%)"
    ].round(1)
)

# Sort from the highest to the lowest mean rating
riepilogo_valutazioni = (
    riepilogo_valutazioni
    .sort_values("Mean", ascending=False)
    .reset_index(drop=True)
)

riepilogo_valutazioni

### Summary of ratings by thematic area

The individual statements are grouped into thematic dimensions to provide a more concise overview of the tourist experience.

The areas concern quality and hospitality, cultural engagement, responsibility towards the area, connection with the place, and perceived well-being.

For each area, the mean score of its component statements is calculated.


In [ ]:
# Associate each area with its corresponding statements
aree_valutazione = {
    "Qualità e accoglienza": [
        "L’esperienza complessiva è stata di alta qualità",
        "L'esperienza di visita mi ha fatto sentire accolto/a",
        "Gli spazi sono stati chiari e accessibili",
        "L’esperienza è stata autentica"
    ],

    "Apprendimento e coinvolgimento culturale": [
        "Le attività mi hanno aiutato a comprendere il territorio",
        "Ho imparato qualcosa di nuovo",
        "L’esperienza culturale mi ha coinvolto/a, non solo come spettatore/trice",
        "Le storie e i luoghi mi hanno colpito emotivamente",
        "L’esperienza mi ha fatto riflettere"
    ],

    "Responsabilità e sostenibilità": [
        "Ho rispettato i luoghi e le persone",
        "Sono stato/a attento/a all’impatto delle mie azioni",
        "L’esperienza mi ha reso più consapevole del valore del territorio",
        "Mi sentirei responsabile nel tornare come visitatore",
        "Sento di aver contribuito positivamente al territorio"
    ],

    "Relazione con il territorio": [
        "Ho interagito con persone del luogo",
        "Ho utilizzato servizi o prodotti locali",
        "Ho vissuto un senso di connessione con il luogo"
    ],

    "Benessere e intenzione di ritorno": [
        "Consiglierei questa esperienza ad altri",
        "Tornerei o rimarrei più a lungo",
        "L’esperienza mi ha fatto stare bene",
        "Mi sono sentito/a rilassato/a e stimolato/a",
        "Questa esperienza ha arricchito il mio tempo",
        "Me ne vado con una sensazione positiva"
    ]
}

# Use the previously calculated table to obtain the mean for each area
riepilogo_aree = []

for area, affermazioni in aree_valutazione.items():
    dati_area = riepilogo_valutazioni[
        riepilogo_valutazioni["Statement"].isin(affermazioni)
    ]

    riepilogo_aree.append({
        "Thematic area": area,
        "Number of statements": len(dati_area),
        "Mean score": dati_area["Mean"].mean(),
        "Positive ratings (%)":
            dati_area["Positive ratings (%)"].mean()
    })

riepilogo_aree = pd.DataFrame(riepilogo_aree)

riepilogo_aree["Mean score"] = (
    riepilogo_aree["Mean score"].round(2)
)

riepilogo_aree["Positive ratings (%)"] = (
    riepilogo_aree["Positive ratings (%)"].round(1)
)

riepilogo_aree = riepilogo_aree.sort_values(
    "Mean score",
    ascending=False
).reset_index(drop=True)

riepilogo_aree

### Distribution of visitors by spending range

Reported expenditure is presented using all spending ranges included in the questionnaire. Categories with no responses are retained in the table to facilitate comparison with future surveys.


In [ ]:
# Complete order of spending ranges
fasce_spesa = [
    "Meno di 100 €",
    "Tra 100 € e 250 €",
    "Tra 250 € e 500 €",
    "Tra 500 € e 1.000 €",
    "Oltre 1.000 €"
]

# Count responses, including ranges with zero responses
conteggio_spesa = (
    df_turisti_pulito[colonna_spesa]
    .value_counts()
    .reindex(fasce_spesa, fill_value=0)
)

# Create the summary table
tabella_spesa = conteggio_spesa.reset_index()
tabella_spesa.columns = [
    "Fascia di spesa",
    "Number of responses"
]

# Calculate percentages using the total number of valid responses
tabella_spesa["Percentuale"] = (
    tabella_spesa["Number of responses"]
    / tabella_spesa["Number of responses"].sum()
    * 100
).round(1)

tabella_spesa

### Checking responses with expenditure above €1,000

The four responses in the highest spending range are examined individually and compared with the characteristics of the visit.

This check is intended to determine whether the reported amounts are plausible or may result from data-entry errors.


In [ ]:
# Select questionnaires reporting expenditure above €1,000
spese_elevate = df_turisti_pulito[
    df_turisti_pulito[colonna_spesa] == "Oltre 1.000 €"
].copy()

# Columns useful for checking response consistency
colonne_controllo_spesa = [
    "ID risposta",
    "È la tua prima visita a Montalto delle Marche?",
    "Che tipo di visita hai o stai realizzando?",
    "Qual'è il motivo principale della tua visita?",
    "Con chi hai visitato Montalto delle Marche?",
    "Hai partecipato ad eventi?",
    colonna_spesa,
    "Professione",
    "Cosa ti ha colpito di più di questa esperienza e perché?",
    "Pensieri liberi. C’è qualcos’altro che vorresti aggiungere?"
]

# Keep only columns that are actually present
colonne_controllo_spesa = [
    colonna
    for colonna in colonne_controllo_spesa
    if colonna in spese_elevate.columns
]

# Display the four responses
spese_elevate[colonne_controllo_spesa]

## Perceived visibility of Montalto delle Marche

This section analyses visitors' perceptions of the Metroborgo project's contribution to the national and international visibility of Montalto delle Marche.

The results represent participants' opinions and are not an objective measurement of an actual increase in the area's visibility.


In [ ]:
# Question concerning the visibility of Montalto delle Marche
colonna_visibilita = (
    "Credi che il progetto Metroborgo abbia contribuito a migliorare "
    "la visibilità nazionale e internazionale di Montalto delle Marche?"
)

# Calculate the number of responses and percentages
tabella_visibilita = tabella_percentuali(
    df_turisti_pulito,
    colonna_visibilita
)

tabella_visibilita

# Analysis of the resident group

This section separately analyses residents and regular visitors to Montalto delle Marche.

Before cleaning and analysing the responses, the structure of the DataFrame is checked to identify:

* columns that are actually used;
* completely empty questions;
* partially completed questionnaires;
* variables that are not relevant to the resident group.


In [ ]:
# Check the number of questionnaires and columns
print("Resident DataFrame dimensions:", df_residenti.shape)

## Exploring the columns

All headers in the resident DataFrame are displayed in order to reconstruct the questionnaire structure and distinguish questions addressed to residents from those intended for visitors.


In [ ]:
# Print all headers in the resident DataFrame
for numero, colonna in enumerate(df_residenti.columns, start=1):
    print(f"{numero}. {colonna}")

print("\nTotal number of columns:", len(df_residenti.columns))

## Number of responses by column

For each column, the number of responses and the number of missing values are calculated.

This check makes it possible to identify irrelevant questions, completely empty columns, and sections that were only partially completed.


In [ ]:
# Display the full text of the questions
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

# Create a table of available and missing responses
riepilogo_colonne_residenti = pd.DataFrame({
    "Column": df_residenti.columns,
    "Number of responses": df_residenti.notna().sum().values,
    "Missing values": df_residenti.isna().sum().values
})

riepilogo_colonne_residenti

## Removing columns not relevant to the resident group

The resident DataFrame contains some questions intended exclusively for visitors.

Since these columns contain no responses in the group being analysed, they are removed. No rows or partially completed questionnaires are removed at this stage.


In [ ]:
# Identify completely empty columns
colonne_vuote_residenti = df_residenti.columns[
    df_residenti.isna().all()
].tolist()

# Create a copy, removing only completely empty columns
df_residenti_pulito = df_residenti.drop(
    columns=colonne_vuote_residenti
).copy()

print("Original columns:", df_residenti.shape[1])
print("Completely empty columns removed:",
      len(colonne_vuote_residenti))
print("Remaining columns:", df_residenti_pulito.shape[1])
print("DataFrame dimensions:",
      df_residenti_pulito.shape)

### Checking the coding of the property-value question

The question concerning changes in property value was exported into five separate columns, one for each possible answer.

Before cleaning the data, the coding of the values in these columns is examined.


In [ ]:
# Identify columns concerning changes in property value
colonne_patrimonio_residenti = [
    colonna
    for colonna in df_residenti_pulito.columns
    if colonna.startswith(
        "A seguito degli investimenti realizzati con il progetto Metroborgo, "
        "ritieni che il valore del tuo patrimonio"
    )
]

print("Number of identified columns:",
      len(colonne_patrimonio_residenti))

# Display the values in each column
for colonna in colonne_patrimonio_residenti:
    print("\n" + "=" * 70)
    print(colonna)
    print(df_residenti_pulito[colonna].value_counts(dropna=False))

### Exploring the economic section

Before cleaning the data, the entire questionnaire section concerning the perceived economic effects of the Metroborgo project is examined.

The objective is to distinguish correctly between questions concerning:

* changes in personal or household income;
* changes in property value;
* environmental quality and quality of life;
* satisfaction with investments;
* economic investments made by participants;
* the amount of reported investment.

No variables are modified at this stage.


In [ ]:
# Select the economic section of the questionnaire
colonne_economiche = df_residenti.columns[69:80]

# Print the original number and full name of each column
for indice, colonna in enumerate(colonne_economiche, start=69):
    print(f"{indice}. {colonna}")

In [ ]:
# Display responses in each economic column
for colonna in colonne_economiche:
    print("\n" + "=" * 80)
    print(colonna)
    print(df_residenti[colonna].value_counts(dropna=False))

### Checking missing or multiple property-value responses

The question concerning changes in property value was exported into five separate columns.

The check shows that:

* 73 participants selected one option;
* 48 participants selected no option;
* 2 participants selected two options.

Before reconstructing a single variable, the two questionnaires with multiple responses are examined. Missing responses will instead be retained as unavailable values.


In [ ]:
# Count the property-value options selected in each questionnaire
numero_scelte_patrimonio = (
    df_residenti_pulito[colonne_patrimonio_residenti]
    .eq("Sì")
    .sum(axis=1)
)

numero_scelte_patrimonio.value_counts().sort_index()

In [ ]:
# Identify questionnaires with more than one selected response
indici_patrimonio_multiplo = numero_scelte_patrimonio[
    numero_scelte_patrimonio > 1
].index

# Select the relevant questionnaires
risposte_patrimonio_multiple = df_residenti_pulito.loc[
    indici_patrimonio_multiplo,
    ["ID risposta"] + colonne_patrimonio_residenti
]

# Display only options selected with "Yes"
for _, riga in risposte_patrimonio_multiple.iterrows():
    print("\n" + "=" * 70)
    print("Response ID:", riga["ID risposta"])

    opzioni_selezionate = [
        colonna.split("[", 1)[-1].rstrip("]")
        for colonna in colonne_patrimonio_residenti
        if riga[colonna] == "Sì"
    ]

    for opzione in opzioni_selezionate:
        print("-", opzione)

### Reconstructing the property-value variable

The question concerning changes in property value was exported into five binary columns.

The new variable is reconstructed by retaining:

* the selected category when there is only one response;
* a missing value when no option was selected;
* a specific category for the two submissions that simultaneously indicate “It remained unchanged” and “It increased slightly”.

This preserves ambiguous responses without arbitrarily assigning them to a category.


In [ ]:
# Function for reconstructing the property-value response
def ricostruisci_patrimonio(riga):
    opzioni_selezionate = [
        colonna.split("[", 1)[-1].rstrip("]")
        for colonna in colonne_patrimonio_residenti
        if riga[colonna] == "Sì"
    ]

    # No response selected
    if len(opzioni_selezionate) == 0:
        return pd.NA

    # One response selected
    if len(opzioni_selezionate) == 1:
        return opzioni_selezionate[0]

    # Multiple responses selected
    return "Multiple response: " + " / ".join(opzioni_selezionate)


# Create the new categorical variable
df_residenti_pulito["Cambiamento del valore patrimoniale"] = (
    df_residenti_pulito.apply(
        ricostruisci_patrimonio,
        axis=1
    )
)

# Check the result
df_residenti_pulito[
    "Cambiamento del valore patrimoniale"
].value_counts(dropna=False)

### Removing the original property-value columns

After reconstructing the single variable concerning changes in property value, the five original binary columns are removed.

The new variable retains all available information, including missing responses and the two multiple selections.


In [ ]:
# Remove the five original binary columns
df_residenti_pulito = df_residenti_pulito.drop(
    columns=colonne_patrimonio_residenti
).copy()

# Check the new DataFrame dimensions
print("Updated DataFrame dimensions:",
      df_residenti_pulito.shape)

### Checking open-ended responses

Text responses are examined to identify test questionnaires, irrelevant notes, or submissions that may need to be excluded from the analysis.

Each open-ended response is displayed together with the corresponding questionnaire ID. No rows are removed at this stage.


In [ ]:
# Columns containing residents' open-ended responses
colonne_aperte_residenti = [
    "Cosa ti aspetti che cambi, grazie al progetto Metroborgo, "
    "nella vita del territorio e nella tua vita personale?",
    
    "Pensieri liberi. C’è qualcos’altro che vorresti aggiungere?"
]

# Select questionnaires containing at least one open-ended response
risposte_aperte_residenti = df_residenti_pulito.loc[
    df_residenti_pulito[colonne_aperte_residenti]
    .notna()
    .any(axis=1),
    ["ID risposta"] + colonne_aperte_residenti
]

# Print all responses in full
for _, riga in risposte_aperte_residenti.iterrows():
    print("\n" + "=" * 80)
    print("Response ID:", riga["ID risposta"])

    for colonna in colonne_aperte_residenti:
        if pd.notna(riga[colonna]):
            print(f"\n{colonna}")
            print("Response:", riga[colonna])

### Removing technical columns

Technical variables generated by the data-collection system and the question used to divide the sample are removed from the resident DataFrame.

The response ID is retained to allow checks on individual questionnaires.


In [ ]:
# Technical columns not required for the analysis
colonne_tecniche_residenti = [
    "Data invio",
    "Ultima pagina",
    "Lingua iniziale",
    "Seme",
    "Data di inizio",
    "Data dell'ultima azione",
    colonna_residenza
]

# Keep only columns that are actually present
colonne_tecniche_residenti = [
    colonna
    for colonna in colonne_tecniche_residenti
    if colonna in df_residenti_pulito.columns
]

# Remove technical columns
df_residenti_pulito = df_residenti_pulito.drop(
    columns=colonne_tecniche_residenti
).copy()

print("Technical columns removed:",
      len(colonne_tecniche_residenti))

print("Updated dimensions:",
      df_residenti_pulito.shape)

### Checking questionnaire completion levels

For each questionnaire, the number of answers provided to the main questions is calculated.

Identification variables, open-ended questions, and some conditional questions that may legitimately be blank are excluded from the count.

This check helps identify completely empty or interrupted submissions before proceeding with the analysis.


In [ ]:
# Columns to exclude from the completion check
colonne_escluse_completamento = [
    "ID risposta",
    "Cosa ti aspetti che cambi, grazie al progetto Metroborgo, "
    "nella vita del territorio e nella tua vita personale?",
    "Pensieri liberi. C’è qualcos’altro che vorresti aggiungere?",
    "Professione [Altro]",
    "Qual è stato approssimativamente l’ammontare complessivo "
    "degli investimenti effettuati?"
]

# Keep only columns that are actually present
colonne_escluse_completamento = [
    colonna
    for colonna in colonne_escluse_completamento
    if colonna in df_residenti_pulito.columns
]

# Identify the main columns to consider
colonne_analisi_residenti = [
    colonna
    for colonna in df_residenti_pulito.columns
    if colonna not in colonne_escluse_completamento
]

# Calculate the number of completed responses per questionnaire
completamento_residenti = pd.DataFrame({
    "ID risposta": df_residenti_pulito["ID risposta"],
    "Completed responses": (
        df_residenti_pulito[colonne_analisi_residenti]
        .notna()
        .sum(axis=1)
    )
})

# Also calculate the number of missing responses
completamento_residenti["Missing responses"] = (
    len(colonne_analisi_residenti)
    - completamento_residenti["Completed responses"]
)

# Sort from the least complete to the most complete questionnaires
completamento_residenti = completamento_residenti.sort_values(
    by="Completed responses"
)

completamento_residenti

### Removing questionnaires without useful responses

The completion-level check identified 40 questionnaires containing no answers to the main questions addressed to residents.

These rows are excluded from the analysis. Partially completed questionnaires are temporarily retained and examined separately before deciding whether to use them.


In [ ]:
# Identify IDs of questionnaires without main responses
id_residenti_vuoti = completamento_residenti.loc[
    completamento_residenti["Completed responses"] == 0,
    "ID risposta"
].tolist()

print("Completely empty questionnaires:",
      len(id_residenti_vuoti))

# Remove only completely empty questionnaires
df_residenti_pulito = df_residenti_pulito[
    ~df_residenti_pulito["ID risposta"].isin(id_residenti_vuoti)
].copy()

# Reset the DataFrame index
df_residenti_pulito.reset_index(drop=True, inplace=True)

print("Remaining resident questionnaires:",
      len(df_residenti_pulito))

print("Updated dimensions:",
      df_residenti_pulito.shape)

### Analysing partially completed questionnaires

After removing completely empty questionnaires, incomplete submissions are examined.

For each questionnaire, the following are reported:

* the number of main questions completed;
* the last question answered;
* the first missing question.

This check helps determine whether participants completed the main perception sections and stopped only in the economic or sociodemographic sections.


In [ ]:
# Select questionnaires with at least one response,
# but fewer than 40 completed main questions
id_residenti_parziali = completamento_residenti.loc[
    completamento_residenti["Completed responses"].between(1, 39),
    "ID risposta"
].tolist()

# Select the remaining partial questionnaires
questionari_parziali_residenti = df_residenti_pulito[
    df_residenti_pulito["ID risposta"].isin(id_residenti_parziali)
]

# Create a concise summary
riepilogo_parziali_residenti = []

for _, riga in questionari_parziali_residenti.iterrows():

    colonne_compilate = [
        colonna
        for colonna in colonne_analisi_residenti
        if pd.notna(riga[colonna])
    ]

    colonne_mancanti = [
        colonna
        for colonna in colonne_analisi_residenti
        if pd.isna(riga[colonna])
    ]

    riepilogo_parziali_residenti.append({
        "ID risposta": riga["ID risposta"],
        "Completed responses": len(colonne_compilate),
        "Last completed question": (
            colonne_compilate[-1]
            if colonne_compilate
            else None
        ),
        "First missing question": (
            colonne_mancanti[0]
            if colonne_mancanti
            else None
        )
    })

# Convert the summary into a table
riepilogo_parziali_residenti = pd.DataFrame(
    riepilogo_parziali_residenti
).sort_values(
    "Completed responses"
).reset_index(drop=True)

pd.set_option("display.max_colwidth", None)

print(
    "Partial questionnaires identified:",
    len(riepilogo_parziali_residenti)
)

riepilogo_parziali_residenti

### Outcome of the partial-completion check

The 13 partially completed questionnaires are not excluded from the analysis.

Most participants completed the main sections concerning perceptions of the area, skipping only a few questions or stopping in the final economic or sociodemographic sections.

Subsequent analyses therefore use the corresponding number of valid responses for each question. This retains all available information without assigning values to missing responses.


## Sociodemographic profile of residents

This section describes the composition of the sample of residents and regular visitors using the following variables:

* gender;
* age group;
* educational qualification;
* occupation;
* number of years of residence or regular attendance.

Percentages are calculated using the total number of valid responses available for each question.


In [ ]:
# Sociodemographic and length-of-stay variables to analyse
variabili_profilo_residenti = [
    "Genere",
    "Età:",
    "Titolo di studio",
    "Professione",
    "Da quanti anni vivi a/frequenti abitualmente Montalto delle Marche?"
]

# Create and display a table for each variable
for variabile in variabili_profilo_residenti:
    print("\n" + "=" * 80)
    print(variabile.upper())
    print("=" * 80)

    display(
        tabella_percentuali(
            df_residenti_pulito,
            variabile
        )
    )

## Perceptions of the area and community

This section analyses statements concerning participation, trust, sense of community, connection with the area, culture, sustainability, and perceived well-being.

Responses are given on a scale from 1 to 7.

For each statement, the following are calculated:

* the number of valid responses;
* the mean score;
* the median;
* the percentage of positive ratings, corresponding to scores from 5 to 7.


In [ ]:
# Identify columns containing statements rated on a 1-to-7 scale
colonne_valutazioni_residenti = [
    colonna
    for colonna in df_residenti_pulito.columns
    if colonna.startswith(
        "Quanto sei d’accordo con le seguenti affermazioni?"
    )
]

# Create the ratings summary
riepilogo_valutazioni_residenti = []

for colonna in colonne_valutazioni_residenti:
    risposte = pd.to_numeric(
        df_residenti_pulito[colonna],
        errors="coerce"
    ).dropna()

    # Extract the statement text
    affermazione = colonna.split("[", 1)[-1].rstrip("]")

    riepilogo_valutazioni_residenti.append({
        "Statement": affermazione,
        "Valid responses": len(risposte),
        "Mean": risposte.mean(),
        "Median": risposte.median(),
        "Positive ratings (%)": (
            risposte.ge(5).mean() * 100
        )
    })

# Convert the results into a table
riepilogo_valutazioni_residenti = pd.DataFrame(
    riepilogo_valutazioni_residenti
)

# Round the values
riepilogo_valutazioni_residenti["Mean"] = (
    riepilogo_valutazioni_residenti["Mean"].round(2)
)

riepilogo_valutazioni_residenti["Median"] = (
    riepilogo_valutazioni_residenti["Median"].round(1)
)

riepilogo_valutazioni_residenti[
    "Positive ratings (%)"
] = (
    riepilogo_valutazioni_residenti[
        "Positive ratings (%)"
    ].round(1)
)

# Sort from the highest to the lowest mean
riepilogo_valutazioni_residenti = (
    riepilogo_valutazioni_residenti
    .sort_values("Mean", ascending=False)
    .reset_index(drop=True)
)

riepilogo_valutazioni_residenti

### Summary of perceptions by thematic area

The 25 statements are grouped into six dimensions:

* initiative and participation;
* social cohesion and sense of community;
* identity-based connection with the area;
* cultural participation and engagement;
* environmental responsibility;
* well-being and future prospects.

For each participant, the mean score is calculated for the different areas. A score is considered valid when at least half of the statements in that area have been completed.

The overall mean score and the percentage of positive ratings, corresponding to a mean score of 5 or above, are then calculated.


In [ ]:
import math

# Associate the short statement text
# with the full name of the corresponding column
mappa_colonne_residenti = {
    colonna.split("[", 1)[-1].rstrip("]"): colonna
    for colonna in colonne_valutazioni_residenti
}

# Group the statements into thematic areas
aree_residenti = {
    "Iniziativa e partecipazione": [
        "Sento di poter contribuire attivamente alla vita del mio territorio",
        "Quando vedo un problema o un’opportunità nel paese, penso di poter fare qualcosa",
        "Mi sento ascoltato/a quando esprimo idee o proposte sul futuro del territorio",
        "Ho fiducia nella mia capacità di avviare o partecipare a iniziative locali"
    ],

    "Coesione sociale e comunità": [
        "Nel mio territorio le persone collaborano tra loro",
        "Mi sento parte di una comunità",
        "C’è fiducia reciproca tra abitanti, associazioni e operatori locali"
    ],

    "Legame con il territorio": [
        "Mi sento emotivamente legato/a a questo luogo",
        "Sono orgoglioso/a di vivere qui",
        "Questo territorio rappresenta una parte importante di chi sono",
        "Mi dispiacerebbe molto dover lasciare questo luogo"
    ],

    "Cultura e coinvolgimento": [
        "Partecipo ad attività culturali locali (eventi, laboratori, incontri)",
        "Gli spazi culturali mi stimolano a imparare cose nuove",
        "Le attività culturali mi fanno riflettere sul presente e sul futuro del territorio",
        "Mi sento coinvolto/a, non solo spettatore/trice",
        "La cultura ha un ruolo importante nella mia vita quotidiana"
    ],

    "Responsabilità ambientale": [
        "Presto attenzione alla cura degli spazi pubblici",
        "Penso alle conseguenze future delle scelte fatte oggi sul territorio",
        "Nella mia vita quotidiana presto attenzione a ridurre l’impatto ambientale delle mie azioni nel territorio in cui vivo",
        "Cerco di adottare comportamenti rispettosi dell’ambiente negli spazi pubblici e culturali (rifiuti, energia, mobilità, uso delle risorse)"
    ],

    "Benessere e prospettive future": [
        "Mi sento soddisfatto/a della mia vita nel territorio",
        "Vivo questo luogo come un ambiente che favorisce il benessere",
        "Le relazioni sociali contribuiscono positivamente alla mia qualità della vita",
        "Vedo prospettive positive per il futuro della comunità",
        "Mi sento mentalmente ed emotivamente a mio agio qui"
    ]
}

# Create a DataFrame that will contain the score
# obtained by each participant in the different areas
punteggi_aree_residenti = pd.DataFrame(
    index=df_residenti_pulito.index
)

for area, affermazioni in aree_residenti.items():

    # Retrieve the full column names
    colonne_area = [
        mappa_colonne_residenti[affermazione]
        for affermazione in affermazioni
    ]

    # Convert responses into numeric values
    valori_area = df_residenti_pulito[
        colonne_area
    ].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Require at least half of the statements to be completed
    minimo_risposte = math.ceil(len(colonne_area) / 2)

    # Calculate the individual mean for the area
    punteggio_area = valori_area.mean(axis=1)

    # Set the score as missing for participants who
    # did not answer at least half of the statements
    punteggio_area[
        valori_area.notna().sum(axis=1) < minimo_risposte
    ] = pd.NA

    punteggi_aree_residenti[area] = punteggio_area


# Create the overall summary of the areas
riepilogo_aree_residenti = []

for area in punteggi_aree_residenti.columns:
    punteggi_validi = pd.to_numeric(
        punteggi_aree_residenti[area],
        errors="coerce"
    ).dropna()

    riepilogo_aree_residenti.append({
        "Thematic area": area,
        "Number of statements": len(aree_residenti[area]),
        "Valid responses": len(punteggi_validi),
        "Mean score": punteggi_validi.mean(),
        "Median": punteggi_validi.median(),
        "Positive ratings (%)": (
            punteggi_validi.ge(5).mean() * 100
        )
    })

riepilogo_aree_residenti = pd.DataFrame(
    riepilogo_aree_residenti
)

# Round the results
riepilogo_aree_residenti["Mean score"] = (
    riepilogo_aree_residenti["Mean score"].round(2)
)

riepilogo_aree_residenti["Median"] = (
    riepilogo_aree_residenti["Median"].round(2)
)

riepilogo_aree_residenti["Positive ratings (%)"] = (
    riepilogo_aree_residenti[
        "Positive ratings (%)"
    ].round(1)
)

# Sort from the highest to the lowest rating
riepilogo_aree_residenti = (
    riepilogo_aree_residenti
    .sort_values(
        "Mean score",
        ascending=False
    )
    .reset_index(drop=True)
)

riepilogo_aree_residenti

## Perceived effects of the Metroborgo project

This section analyses residents' opinions about the effects of the Metroborgo project.

The analysis considers:

* the perceived improvement in social life;
* dialogue with people from different cultural backgrounds;
* the potential to encourage new cultural, creative, and tourism businesses;
* perceived changes in income;
* environmental quality;
* influence on quality of life;
* satisfaction with investments;
* improved visibility of Montalto delle Marche.

Percentages are calculated using the total number of valid responses available for each question.


In [ ]:
# Keywords sufficient to identify the questions
parole_chiave_effetti = [
    "miglioramento della tua vita sociale",
    "background culturale differente",
    "favorire nuove imprese",
    "come è cambiato il tuo reddito",
    "cambiamento della qualità dell’ambiente",
    "influenzato la qualità della tua vita",
    "soddisfatto/a degli investimenti",
    "visibilità nazionale e internazionale"
]

# Identify the exact column names in the DataFrame
variabili_effetti_metroborgo = []

for parola_chiave in parole_chiave_effetti:
    corrispondenze = [
        colonna
        for colonna in df_residenti_pulito.columns
        if parola_chiave.lower() in colonna.lower()
    ]

    if len(corrispondenze) == 1:
        variabili_effetti_metroborgo.append(corrispondenze[0])
    else:
        print(
            f"Warning: found {len(corrispondenze)} columns "
            f"for the keyword '{parola_chiave}'"
        )

# Create a frequency and percentage table for each question
for variabile in variabili_effetti_metroborgo:
    print("\n" + "=" * 90)
    print(variabile.upper())
    print("=" * 90)

    display(
        tabella_percentuali(
            df_residenti_pulito,
            variabile
        )
    )

## Property and economic investments

This section analyses:

* perceived changes in property value;
* economic investments made in the municipality;
* the amount invested.

Percentages are calculated using the total number of valid responses available for each question.


In [ ]:
# Variables concerning property and investments
variabili_patrimonio_investimenti = [
    "Cambiamento del valore patrimoniale",
    "A seguito del progetto Metroborgo, hai effettuato investimenti economici nel comune di Montalto delle Marche?",
    "Qual è stato approssimativamente l’ammontare complessivo degli investimenti effettuati?"
]

# Create a frequency and percentage table
# for each variable
for variabile in variabili_patrimonio_investimenti:
    print("\n" + "=" * 90)
    print(variabile.upper())
    print("=" * 90)

    display(
        tabella_percentuali(
            df_residenti_pulito,
            variabile
        )
    )